# Lezione 7 — NLP II: Analisi del sentimento con RNN e LSTM

Corso **"Fondamenti di TensorFlow da zero"** — DIH DATAlife

## Obiettivi di questa lezione

- Comprendere perché le RNN (reti neurali ricorrenti) sono adatte al testo sequenziale
- Costruire un modello LSTM (Long Short-Term Memory) per l'analisi del sentimento
- Incorporare le parole in uno spazio denso con embedding
- Interpretare il ruolo della memoria cellulare e del cancello oblio negli LSTM

In [ ]:
import tensorflow as tf
print("Versión de TensorFlow:", tf.__version__)

## 1. Classificatore di sentimento con embedding

Partiamo dal dataset di titoli di notizie etichettati come positivi o negativi. Useremo embedding preaddestrati da TensorFlow Hub, che trasferiscono la conoscenza semantica da miliardi di parole.

In [ ]:
import json
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

!wget --no-check-certificate \
    https://storage.googleapis.com/learning-datasets/sarcasm.json \
    -O /tmp/sarcasm.json

In [ ]:
vocab_size = 10000
embedding_dim = 16
max_length = 100
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"
training_size = 20000

with open("/tmp/sarcasm.json", 'r') as f:
    datastore = json.load(f)

sentences = [item['headline'] for item in datastore]
labels = [item['is_sarcastic'] for item in datastore]

training_sentences = sentences[0:training_size]
testing_sentences = sentences[training_size:]
training_labels = np.array(labels[0:training_size])
testing_labels = np.array(labels[training_size:])

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(training_sentences)

training_sequences = tokenizer.texts_to_sequences(training_sentences)
training_padded = pad_sequences(training_sequences, maxlen=max_length,
                                 padding=padding_type, truncating=trunc_type)

testing_sequences = tokenizer.texts_to_sequences(testing_sentences)
testing_padded = pad_sequences(testing_sequences, maxlen=max_length,
                                padding=padding_type, truncating=trunc_type)

print(training_padded.shape, testing_padded.shape)

In [ ]:
sentiment_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

sentiment_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
sentiment_model.summary()

In [ ]:
# Recortamos a 10 épocas para que la celda se ejecute rápido en clase
# (en el manual se entrena con 30 épocas y se observa más sobreajuste)
history = sentiment_model.fit(
    training_padded, training_labels,
    epochs=10,
    validation_data=(testing_padded, testing_labels),
    verbose=2
)

In [ ]:
import matplotlib.pyplot as plt

def plot_graphs(history, string):
    plt.plot(history.history[string])
    plt.plot(history.history['val_' + string])
    plt.xlabel("Epochs")
    plt.ylabel(string)
    plt.legend([string, 'val_' + string])
    plt.show()

plot_graphs(history, "accuracy")
plot_graphs(history, "loss")

In [ ]:
nuevas_frases = [
    "granny starting to fear spiders in the garden might be real",
    "game of thrones season finale showing this sunday night"
]
secuencias = tokenizer.texts_to_sequences(nuevas_frases)
padded = pad_sequences(secuencias, maxlen=max_length, padding=padding_type, truncating=trunc_type)
print(sentiment_model.predict(padded))

## 2. Riutilizzare embedding preaddestrati con TensorFlow Hub

Invece di addestrare i propri embedding da zero, puoi riutilizzare embedding preaddestrati (da miliardi di parole) da TensorFlow Hub. Questo trasferisce la conoscenza di relazioni semantiche apprese su testi enormi al tuo compito specifico.

In [ ]:
!pip install -q tensorflow-hub tensorflow-datasets

In [ ]:
import tensorflow_hub as hub
import tensorflow_datasets as tfds

# 60% train / 40% validación, más el split de test oficial
train_data, validation_data, test_data = tfds.load(
    name="imdb_reviews",
    split=('train[:60%]', 'train[60%:]', 'test'),
    as_supervised=True
)

train_examples_batch, train_labels_batch = next(iter(train_data.batch(3)))
train_examples_batch

In [ ]:
embedding_url = "https://tfhub.dev/google/nnlm-en-dim50/2"
hub_layer = hub.KerasLayer(embedding_url, input_shape=[], dtype=tf.string, trainable=True)

# Comprobación rápida: cualquier frase se convierte en un vector de 50 valores
hub_layer(train_examples_batch[:3])

In [ ]:
hub_model = tf.keras.Sequential([
    hub_layer,
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1)
])

hub_model.compile(optimizer='adam',
                   loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
                   metrics=['accuracy'])
hub_model.summary()

In [ ]:
# Recortamos a 3 épocas para la clase (el tutorial original usa 10)
hub_history = hub_model.fit(
    train_data.shuffle(10000).batch(512),
    epochs=3,
    validation_data=validation_data.batch(512),
    verbose=1
)

results = hub_model.evaluate(test_data.batch(512), verbose=2)
for name, value in zip(hub_model.metrics_names, results):
    print("%s: %.3f" % (name, value))

## 3. Perché una rete densa non basta per dati sequenziali: la RNN

`GlobalAveragePooling1D` appiattisce una sequenza in un unico vettore calcolando la media. Funziona, ma perde l'ordine: "I love this movie" produce lo stesso vettore di "movie love I this". Le RNN processano la sequenza parola per parola, mantenendo memoria di ciò che hanno visto prima.

## 4. LSTM: memoria a lungo e corto termine

LSTM aggiunge uno **stato di cella** che accumula informazioni nel tempo. A differenza di una RNN semplice, LSTM ha cancelli che controllano cosa ricordare e cosa dimenticare, evitando il problema del gradiente che svanisce (una RNN semplice perde memoria dei token all'inizio dopo pochi passi).

In [ ]:
lstm_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_model.summary()

lstm_history = lstm_model.fit(
    training_padded, training_labels,
    epochs=10,
    validation_data=(testing_padded, testing_labels),
    verbose=2
)

Confronta la colonna `val_accuracy` di questo addestramento con quella del modello con `GlobalAveragePooling`: LSTM dovrebbe raggiungere una precisione più alta.

## 5. Generare testo nuovo (poesia) con LSTM

Cambiamo compito: invece di classificare, generiamo testo nuovo. Alleniamo un LSTM a carattere per carattere su una collezione di poesie e gli chiediamo di continuare una frase iniziale.

In [ ]:
!wget --no-check-certificate \
    https://storage.googleapis.com/learning-datasets/irish-lyrics-eof.txt \
    -O /tmp/irish-lyrics-eof.txt

In [ ]:
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

poem_tokenizer = Tokenizer()
data = open('/tmp/irish-lyrics-eof.txt').read()
corpus = data.lower().split("\n")

poem_tokenizer.fit_on_texts(corpus)
total_words = len(poem_tokenizer.word_index) + 1
print("Palabras en el vocabulario:", total_words)

In [ ]:
input_sequences = []
for line in corpus:
    token_list = poem_tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i + 1])

max_sequence_len = max(len(seq) for seq in input_sequences)
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

xs, labels_poem = input_sequences[:, :-1], input_sequences[:, -1]
ys = tf.keras.utils.to_categorical(labels_poem, num_classes=total_words)

print("Forma de xs:", xs.shape, "- Forma de ys:", ys.shape)

In [ ]:
# Hiperparámetros reducidos (embedding y unidades LSTM más pequeños, menos
# épocas) para que el entrenamiento sea rápido en clase. El tutorial original
# usa 100 unidades de embedding, 150 unidades LSTM y 100 épocas.
poem_model = Sequential([
    Embedding(total_words, 64, input_length=max_sequence_len - 1),
    Bidirectional(LSTM(64)),
    Dense(total_words, activation='softmax')
])

poem_model.compile(loss='categorical_crossentropy',
                    optimizer=Adam(learning_rate=0.01),
                    metrics=['accuracy'])
poem_model.summary()

In [ ]:
poem_history = poem_model.fit(xs, ys, epochs=30, verbose=1)

In [ ]:
plot_graphs(poem_history, 'accuracy')

In [ ]:
def generar_texto(seed_text, next_words, model, tokenizer, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

seed_text = "I've got a bad feeling about this"
print(generar_texto(seed_text, 20, poem_model, poem_tokenizer, max_sequence_len))

## Riepilogo della lezione

- Un classificatore di sentimento semplice: Embedding + GlobalAveragePooling + Dense
- Embedding preaddestrati da TensorFlow Hub trasferiscono conoscenza semantica
- RNN e LSTM processano sequenze mantenendo memoria dello stato precedente
- LSTM aggiunge cancelli per evitare il problema del gradiente che svanisce
- Dopo la classificazione, le RNN possono generare testo nuovo autoregressivamente